In [1]:
from __future__ import annotations

import os
import hashlib
import json
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List

# ----------------------------
# Windows/HF cache tweaks (set BEFORE Docling imports)
# ----------------------------
# Put HF cache somewhere inside your project (optional but tidy)
BASE = Path.cwd()
HF_HOME = BASE / ".hf_cache"
os.environ.setdefault("HF_HOME", str(HF_HOME))

# Warning suppression (WinError 1314)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# ----------------------------
# Docling imports + PDF pipeline options (disable OCR)
# ----------------------------
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractCliOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

C:\Users\tidemanlem\Documents\Course_Alexey_Grigorev\MyAgent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ----------------------------
# Data model / utilities
# ----------------------------
@dataclass
class RagDoc:
    doc_id: str
    source: str  # local path
    title: str
    text: str
    fetched_at_utc: str
    sha256: str
    meta: Dict[str, Any]

def utc_now_iso() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", errors="ignore")).hexdigest()

def safe_slug(s: str, max_len: int = 80) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "-", s).strip("-")
    return s[:max_len] if s else "doc"

def make_doc_id(prefix: str, source: str, content_hash: str) -> str:
    slug = safe_slug(source)
    short = content_hash[:12] if content_hash else "nohash"
    return f"{prefix}-{slug}-{short}"

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def write_doc(out_dir: Path, doc: RagDoc) -> None:
    json_path = out_dir / f"{doc.doc_id}.json"
    txt_path = out_dir / f"{doc.doc_id}.txt"

    with json_path.open("w", encoding="utf-8") as f:
        json.dump(asdict(doc), f, ensure_ascii=False, indent=2)

    with txt_path.open("w", encoding="utf-8") as f:
        f.write(doc.text or "")

    jsonl_path = out_dir / "docs.jsonl"
    with jsonl_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(asdict(doc), ensure_ascii=False) + "\n")

def collect_pdfs(root: Path) -> List[Path]:
    if not root.exists():
        return []
    if root.is_file() and root.suffix.lower() == ".pdf":
        return [root]
    if root.is_dir():
        return sorted(root.rglob("*.pdf"))
    return []

def build_docling_converter_no_ocr() -> DocumentConverter:
    pdf_options = PdfPipelineOptions()
    pdf_options.do_ocr = True  # key change: do not run OCR

    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
        }
    )

def file_fingerprint(p: Path) -> str:
    """
    Cheap + reliable-enough fingerprint for incremental runs:
    absolute path + size + last-modified timestamp.
    If you prefer strongest correctness, compute sha256 of the PDF bytes instead (slower).
    """
    st = p.stat()
    return f"{str(p.resolve())}|{st.st_size}|{int(st.st_mtime)}"

def load_manifest(out_dir: Path) -> Dict[str, Any]:
    manifest_path = out_dir / "manifest.json"
    if manifest_path.exists():
        return json.loads(manifest_path.read_text(encoding="utf-8"))
    return {"version": 1, "files": {}}  # files[fingerprint] = {doc_id, source, ...}

def save_manifest(out_dir: Path, manifest: Dict[str, Any]) -> None:
    manifest_path = out_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

In [3]:
def build_docling_converter_no_ocr() -> DocumentConverter:
    pdf_options = PdfPipelineOptions()
    pdf_options.do_ocr = False

    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
        }
    )

def build_docling_converter_with_ocr() -> DocumentConverter:
    ocr_options = TesseractCliOcrOptions(
        lang=["eng"],  # use ["auto"] later if needed
        tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe",
        path=r"C:\Program Files\Tesseract-OCR\tessdata",  # optional but often helpful
    )

    pdf_options = PdfPipelineOptions(
        do_ocr=True,
        force_full_page_ocr=True,
        ocr_options=ocr_options,
    )

    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
        }
    )

def looks_like_image_only_output(text: str) -> bool:
    if not text or not text.strip():
        return True

    compact = re.sub(r"\s+", "", text)
    if re.fullmatch(r"(<!--image-->)+", compact):
        return True

    # not much real text
    return len(re.findall(r"[A-Za-z0-9]", text)) < 100

def extract_best_text(pdf_path: Path,
                      fast_converter: DocumentConverter,
                      ocr_converter: DocumentConverter) -> tuple[str, list[str], str]:
    # First try without OCR
    result = fast_converter.convert(pdf_path, raises_on_error=False)
    errors = [str(e) for e in (getattr(result, "errors", None) or [])]
    doc_obj = getattr(result, "document", None)

    if doc_obj is not None:
        content = (doc_obj.export_to_text() or "").strip()
        if not looks_like_image_only_output(content):
            return content, errors, "no_ocr"

    # Fallback to OCR
    result = ocr_converter.convert(pdf_path, raises_on_error=False)
    errors = [str(e) for e in (getattr(result, "errors", None) or [])]
    doc_obj = getattr(result, "document", None)

    if doc_obj is None:
        return "", errors, "ocr_failed"

    content = (doc_obj.export_to_text(traverse_pictures=True) or "").strip()
    return content, errors, "ocr"

In [4]:
def ingest_pdfs_docling(
    pdf_paths: Iterable[Path],
    out_dir: Path,
) -> None:
    ensure_dir(out_dir)

    manifest = load_manifest(out_dir)
    seen: Dict[str, Any] = manifest.setdefault("files", {})

    fast_converter = build_docling_converter_no_ocr()
    ocr_converter = build_docling_converter_with_ocr()

    for pdf_path in pdf_paths:
        pdf_path = pdf_path.resolve()
        fp = file_fingerprint(pdf_path)

        if fp in seen:
            print(f"[skip] {pdf_path.name}")
            continue

        try:
            content, errors, mode = extract_best_text(pdf_path, fast_converter, ocr_converter)
        except Exception as e:
            print(f"[error] {pdf_path} → {e}")
            continue

        if not content:
            print(f"[empty] {pdf_path} (no extractable text even after OCR)")
            continue

        text_hash = sha256_text(content)
        slug_base = f"{pdf_path.parent.name}-{pdf_path.stem}"
        doc_id = make_doc_id("pdf", slug_base, text_hash)

        doc = RagDoc(
            doc_id=doc_id,
            source=str(pdf_path),
            title=pdf_path.stem,
            text=content,
            fetched_at_utc=utc_now_iso(),
            sha256=text_hash,
            meta={
                "file_name": pdf_path.name,
                "file_size": pdf_path.stat().st_size,
                "mtime": int(pdf_path.stat().st_mtime),
                "docling_errors": errors,
                "extraction_mode": mode,
            },
        )

        write_doc(out_dir, doc)

        seen[fp] = {
            "doc_id": doc_id,
            "source": str(pdf_path),
            "file_name": pdf_path.name,
            "file_size": pdf_path.stat().st_size,
            "mtime": int(pdf_path.stat().st_mtime),
            "sha256_text": text_hash,
            "written_at_utc": utc_now_iso(),
            "extraction_mode": mode,
        }
        save_manifest(out_dir, manifest)

        print(f"[pdf] wrote {doc_id} via {mode}")

In [5]:
base = Path.cwd()
pdfs_root = base / "pdfs" / "new"
out_dir = base / "data" / "granite_docling_pdfs"

pdf_paths = collect_pdfs(pdfs_root)
print(f"Found {len(pdf_paths)} PDFs under: {pdfs_root}")

converter = build_docling_converter_no_ocr()
ingest_pdfs_docling(pdf_paths, out_dir)

Found 10 PDFs under: C:\Users\tidemanlem\Documents\Course_Alexey_Grigorev\MyAgent\pdfs\new


Parameter `strict_text` has been deprecated and will be ignored.


[pdf] wrote pdf-new-ai-agents-under-eu-law-5bb0fb6a6aa8 via no_ocr


Parameter `strict_text` has been deprecated and will be ignored.


[pdf] wrote pdf-new-kauffman-navigating-the-eu-ai-act-a-practical-guide-for-global-manufacturing-5d5431ae6958 via no_ocr


...

[pdf] wrote pdf-new-national-culture-and-ai-governance-the-cultural-origins-of-differences-in-ai-bbd147567735 via no_ocr


Parameter `strict_text` has been deprecated and will be ignored.


[pdf] wrote pdf-new-scalable-runtime-governance-for-agentic-ai-in-financial-services-lukasz-szpr-ca0f8f2a8922 via no_ocr


Parameter `strict_text` has been deprecated and will be ignored.


[error] C:\Users\tidemanlem\Documents\Course_Alexey_Grigorev\MyAgent\pdfs\new\Thou Shalt Pay - Luiza Jarovsky.pdf → Tesseract is not available, aborting: [WinError 2] The system cannot find the file specified Install tesseract on your system and the tesseract binary is discoverable. The actual command for Tesseract can be specified in `pipeline_options.ocr_options.tesseract_cmd='tesseract'`. Alternatively, Docling has support for other OCR engines. See the documentation.
